Module 01: Exploratory Data Analysis for Demand & Inventory with Polars

This notebook performs exploratory data analysis (EDA) for Module 01 of the **"Intelligent System for Supply Chain Management"** project.  

The primary goal is to optimize inventory and purchasing management, with a target of **reducing overstocking by 20%** within six months.

---

## Import Libraries

In [2]:
import polars as pl
import polars.selectors as cs
import json
import plotly.express as px
import plotly.io as pio
from pathlib import Path
from polars_info import print_df_info
from datetime import date

# Install dependencies as needed:
# pip install kagglehub[polars-datasets]
import kagglehub
from kagglehub import KaggleDatasetAdapter

import warnings
warnings.filterwarnings('ignore')

# Set up display options and plotting template
pio.templates.default = "plotly_white"
px.defaults.width = 800
px.defaults.height = 600

## Load Dataset

In [3]:
# Paths
path_docs = Path("..") / "docs"
path_processed = Path("..") / "data" / "processed"

In [4]:
# Set the path to the file you'd like to load
file_path = "1M_grocery_data_pl.parquet"

# Load the latest version
lf = kagglehub.dataset_load(
  KaggleDatasetAdapter.POLARS,
  "robertobalbinotti/synthetic-grocery-data",
  file_path,
  # Provide any additional arguments like
  # sql_query, polars_frame_type, or 
  # polars_kwargs.
  # See the documenation for more information:
  # https://github.com/Kaggle/kagglehub/blob/main/README.md#kaggledatasetadapterpolars
)

In [9]:
# Load column descriptions from JSON file into a dictionary for reference or documentation
with open(path_docs / 'column_descriptions_polars.json') as f:
    column_descriptions = json.load(f)

# Data Cleaning and Preprocessing

In [11]:
print_df_info(lf.collect())

<class 'polars.dataframe.frame.DataFrame'>
Shape: (1,000,658, 31)
Estimated size: 182.25 MiB
Columns:
  #  Column                        Dtype         Non-Null    Null   Null%
  0  order_purchase_date           Date          1,000,658       0   0.00%
  1  received_date                 Date          1,000,658       0   0.00%
  2  product_id                    String        1,000,658       0   0.00%
  3  product                       String        1,000,658       0   0.00%
  4  category                      String        1,000,658       0   0.00%
  5  sub_category                  String        1,000,658       0   0.00%
  6  sales_demand                  String        1,000,658       0   0.00%
  7  sales_volume                  UInt16        1,000,658       0   0.00%
  8  seasonality                   List(String)  1,000,658       0   0.00%
  9  storage_recommendation        String        1,000,658       0   0.00%
 10  unit_of_measurement           String        1,000,658       0   0.00%

DFInfoSummary(rows=1000658, cols=31, estimated_size_bytes=191102740, dtypes={'order_purchase_date': Date, 'received_date': Date, 'product_id': String, 'product': String, 'category': String, 'sub_category': String, 'sales_demand': String, 'sales_volume': UInt16, 'seasonality': List(String), 'storage_recommendation': String, 'unit_of_measurement': String, 'shelf_life_days': UInt16, 'maximum_days_on_sale': UInt16, 'supplier_id': String, 'supplier': String, 'supplier_rating': UInt8, 'distance_km': UInt16, 'moq': UInt16, 'delivery_days': Float16, 'transit_time': Float16, 'in_season': Boolean, 'is_holiday': Boolean, 'day_classification': String, 'is_weekend': Boolean, 'min_stock': UInt16, 'max_stock': UInt16, 'stock_quantity': UInt16, 'temperature_classification': String, 'precipitation_classification': String, 'wind_classification': String, 'weather_severity': String})

In [12]:
df = lf.with_columns(
    pl.col(pl.Utf8).cast(pl.Categorical()),
).collect()

In [14]:
received_date = df.select("received_date").to_series().sort().unique()

expected_date_range = pl.date_range(
    start= received_date.min(),
    end= received_date.max(),
    interval="1d",
    eager=True
)

is_complete = received_date.len() == expected_date_range.len()
missing_dates = expected_date_range.filter(~expected_date_range.is_in(received_date))

print("Complete Received Date Range?\n", is_complete)
missing_dates

Complete Received Date Range?
 True


literal
date


In [26]:
df.filter(
    pl.struct(['order_purchase_date', 'received_date', 'product_id', 'supplier_id']).is_duplicated()
)

order_purchase_date,received_date,product_id,product,category,sub_category,sales_demand,sales_volume,seasonality,storage_recommendation,unit_of_measurement,shelf_life_days,maximum_days_on_sale,supplier_id,supplier,supplier_rating,distance_km,moq,delivery_days,transit_time,in_season,is_holiday,day_classification,is_weekend,min_stock,max_stock,stock_quantity,temperature_classification,precipitation_classification,wind_classification,weather_severity
date,date,cat,cat,cat,cat,cat,u16,list[str],cat,cat,u16,u16,cat,cat,u8,u16,u16,f16,f16,bool,bool,cat,bool,u16,u16,u16,cat,cat,cat,cat
2022-12-07,2022-12-09,"""1169187|P""","""Tomato""","""Fresh Foods""","""Vegetables""","""High""",164,"[""June"", ""July"", … ""September""]","""Room Temperature""","""lb""",7,3,"""1194877|S""","""ValleyFresh Farms""",4,85,100,0.605469,2.4140625,false,false,"""Weekday""",false,269,369,292,"""Warm""","""No precipitation""","""Gentle to Fresh Breeze""","""Moderate"""
2022-12-06,2022-12-09,"""1741974|P""","""Mozzarella Cheese""","""Dairy & Alternatives""","""Dairy""","""High""",108,[],"""Refrigerated""","""lb""",14,5,"""1422853|S""","""Artisan Cheesemakers""",5,95,40,0.931641,2.769531,false,false,"""Weekday""",false,237,277,275,"""Warm""","""No precipitation""","""Gentle to Fresh Breeze""","""Moderate"""
2022-12-06,2022-12-09,"""1113278|P""","""Carrot""","""Fresh Foods""","""Vegetables""","""High""",163,"[""January"", ""February"", … ""December""]","""Refrigerated""","""lb""",21,7,"""1194877|S""","""ValleyFresh Farms""",4,85,100,1.4140625,2.173828,true,false,"""Weekday""",false,269,369,282,"""Warm""","""No precipitation""","""Gentle to Fresh Breeze""","""Moderate"""
2022-12-06,2022-12-09,"""1202127|P""","""Pomegranate""","""Fresh Foods""","""Fruits""","""High""",168,"[""September"", ""October"", … ""December""]","""Room Temperature""","""unit""",21,7,"""1211763|S""","""Tropical Fruits Ltd.""",4,350,80,1.963867,19.859375,true,false,"""Weekday""",false,439,519,500,"""Warm""","""No precipitation""","""Gentle to Fresh Breeze""","""Moderate"""
2022-12-07,2022-12-09,"""1821093|P""","""Grape Jelly""","""Pantry""","""Condiments""","""High""",30,[],"""Room Temperature""","""unit""",365,90,"""1979851|S""","""Spreadables Inc.""",2,88,45,0.50293,2.0234375,false,false,"""Weekday""",false,70,115,109,"""Warm""","""No precipitation""","""Gentle to Fresh Breeze""","""Moderate"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
2025-09-19,2025-09-22,"""1230940|P""","""Haddock""","""Fresh Foods""","""Seafood""","""Normal""",6,[],"""Refrigerated""","""lb""",2,1,"""1891168|S""","""OceanHarvest Seafood""",2,180,40,0.788086,3.341797,false,false,"""Weekday""",false,23,63,118,"""Mild to Temperate""","""Heavy Rain""","""Gentle to Fresh Breeze""","""Severe"""
2025-09-18,2025-09-22,"""1385860|P""","""Granola Bars""","""Pantry""","""Snacks""","""Normal""",78,[],"""Room Temperature""","""unit""",90,30,"""1783257|S""","""SnackTime Distributors""",5,80,110,1.323242,2.681641,false,false,"""Weekday""",false,203,313,0,"""Mild to Temperate""","""Heavy Rain""","""Gentle to Fresh Breeze""","""Severe"""
2025-09-20,2025-09-22,"""1543068|P""","""Rye Bread""","""Bakery""","""Bread""","""Normal""",109,[],"""Room Temperature""","""unit""",5,2,"""1322480|S""","""Bakery Fresh Co.""",2,45,75,1.256836,1.758789,false,false,"""Weekday""",false,193,268,252,"""Mild to Temperate""","""Heavy Rain""","""Gentle to Fresh Breeze""","""Severe"""


In [31]:
df.filter(
    ~pl.struct([
        'order_purchase_date', 
        'received_date', 
        'product_id', 
        'supplier_id'
    ]).is_duplicated()
)

order_purchase_date,received_date,product_id,product,category,sub_category,sales_demand,sales_volume,seasonality,storage_recommendation,unit_of_measurement,shelf_life_days,maximum_days_on_sale,supplier_id,supplier,supplier_rating,distance_km,moq,delivery_days,transit_time,in_season,is_holiday,day_classification,is_weekend,min_stock,max_stock,stock_quantity,temperature_classification,precipitation_classification,wind_classification,weather_severity
date,date,cat,cat,cat,cat,cat,u16,list[str],cat,cat,u16,u16,cat,cat,u8,u16,u16,f16,f16,bool,bool,cat,bool,u16,u16,u16,cat,cat,cat,cat
2022-12-05,2022-12-09,"""1234660|P""","""Butter""","""Dairy & Alternatives""","""Dairy""","""High""",111,[],"""Refrigerated""","""unit""",30,10,"""1176804|S""","""Daily Dairy""",4,55,85,1.421875,1.375977,false,false,"""Weekday""",false,195,280,216,"""Warm""","""No precipitation""","""Gentle to Fresh Breeze""","""Moderate"""
2022-12-06,2022-12-09,"""1476237|P""","""Canned Tuna""","""Pantry""","""Canned Fish""","""High""",12,[],"""Room Temperature""","""unit""",1095,90,"""1439470|S""","""PantryEssentials Ltd.""",1,95,130,1.018555,2.037109,false,false,"""Weekday""",false,31,161,302,"""Warm""","""No precipitation""","""Gentle to Fresh Breeze""","""Moderate"""
2022-12-05,2022-12-09,"""1457947|P""","""Egg (Quail)""","""Dairy & Alternatives""","""Eggs""","""High""",87,[],"""Refrigerated""","""unit""",28,14,"""1025666|S""","""FreshEggs Co.""",5,65,30,1.12793,1.713867,false,false,"""Weekday""",false,153,183,162,"""Warm""","""No precipitation""","""Gentle to Fresh Breeze""","""Moderate"""
2022-12-06,2022-12-09,"""1659804|P""","""Honey""","""Pantry""","""Sweeteners""","""High""",24,[],"""Room Temperature""","""unit""",1825,365,"""1694005|S""","""Sugar & Spice Co.""",3,85,70,0.896973,2.697266,false,false,"""Weekday""",false,50,120,126,"""Warm""","""No precipitation""","""Gentle to Fresh Breeze""","""Moderate"""
2022-12-08,2022-12-09,"""1606966|P""","""Shrimp""","""Fresh Foods""","""Seafood""","""High""",20,[],"""Refrigerated""","""lb""",2,1,"""1891168|S""","""OceanHarvest Seafood""",2,180,40,0.537598,3.046875,false,false,"""Weekday""",false,25,65,84,"""Warm""","""No precipitation""","""Gentle to Fresh Breeze""","""Moderate"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
2025-09-19,2025-09-22,"""1921785|P""","""Raw Sugar""","""Pantry""","""Baking Supplies""","""Normal""",10,[],"""Room Temperature""","""lb""",730,180,"""1694005|S""","""Sugar & Spice Co.""",3,85,70,1.505859,1.986328,false,false,"""Weekday""",false,24,94,82,"""Mild to Temperate""","""Heavy Rain""","""Gentle to Fresh Breeze""","""Severe"""
2025-09-18,2025-09-22,"""1606966|P""","""Shrimp""","""Fresh Foods""","""Seafood""","""Normal""",6,[],"""Refrigerated""","""lb""",2,1,"""1034692|S""","""Seafood Select""",4,195,28,1.411133,3.671875,false,false,"""Weekday""",false,24,52,40,"""Mild to Temperate""","""Heavy Rain""","""Gentle to Fresh Breeze""","""Severe"""
2025-09-21,2025-09-22,"""1060542|P""","""Strawberries""","""Fresh Foods""","""Fruits""","""Normal""",169,"[""July"", ""August"", … ""November""]","""Refrigerated""","""lb""",4,2,"""1161189|S""","""BerryGood Farms""",5,180,60,0.364258,3.59375,true,false,"""Weekday""",false,363,423,749,"""Mild to Temperate""","""Heavy Rain""","""Gentle to Fresh Breeze""","""Severe"""


In [22]:
df.columns

['order_purchase_date',
 'received_date',
 'product_id',
 'product',
 'category',
 'sub_category',
 'sales_demand',
 'sales_volume',
 'seasonality',
 'storage_recommendation',
 'unit_of_measurement',
 'shelf_life_days',
 'maximum_days_on_sale',
 'supplier_id',
 'supplier',
 'supplier_rating',
 'distance_km',
 'moq',
 'delivery_days',
 'transit_time',
 'in_season',
 'is_holiday',
 'day_classification',
 'is_weekend',
 'min_stock',
 'max_stock',
 'stock_quantity',
 'temperature_classification',
 'precipitation_classification',
 'wind_classification',
 'weather_severity']